# Self-Supervised Deep Representation Learning for Autonomous Detection of Structural Defects in Industrial Systems

**Author:** Final Year Engineering Capstone Project  
**Framework:** PyTorch & Torchvision  
**Hardware:** Google Colab GPU (T4) / Local Machine (CPU or GPU)  

---

### Abstract & Overview
Surface defect inspection is vital in modern manufacturing (steel mills, aerospace, automotive). However, defect images are scarce, and manual annotation is expensive and subjective. This project implements **Self-Supervised Learning (SimCLR and BYOL)** on unlabeled industrial surface images to extract rich geometric and structural representations. We demonstrate that self-supervised pre-training significantly outperforms fully supervised models when labeled data is scarce ($1\%, 5\%, 10\%, 25\%$).

In [ ]:
# 1. Environment & Hardware Verification
import torch
import torchvision
import os
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] Computation Device: {device}")
if device.type == "cuda":
    print(f"[*] GPU Device Name : {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Dataset Preparation & Stratified Partitioning
from data.split_dataset import prepare_splits, create_synthetic_neu_data

raw_dir = Path("data/raw/NEU-DET")
output_splits = Path("data/splits")

# If raw NEU-DET is not yet downloaded, generate synthetic demonstration samples
if not raw_dir.exists():
    print("[*] Generating synthetic sample dataset for pipeline validation...")
    create_synthetic_neu_data(raw_dir, num_per_class=50)

prepare_splits(raw_dir, output_splits, val_ratio=0.15, test_ratio=0.15, seed=42)

In [ ]:
# 3. Visualizing Domain-Specific Industrial Augmentations (Positive Pairs)
from PIL import Image
import matplotlib.pyplot as plt
from data.augmentations import TwoCropsTransform, get_industrial_ssl_transform

sample_img_path = next(Path("data/splits/unlabeled_pretrain/images").glob("*.jpg"))
orig_img = Image.open(sample_img_path).convert("RGB")

two_crops = TwoCropsTransform(get_industrial_ssl_transform(200))
view1, view2 = two_crops(orig_img)

fig, ax = plt.subplots(1, 3, figsize=(10, 4))
ax[0].imshow(orig_img)
ax[0].set_title("Original Surface Scan")
ax[0].axis("off")

v1_np = view1.permute(1, 2, 0).numpy()
v1_np = (v1_np - v1_np.min()) / (v1_np.max() - v1_np.min())
ax[1].imshow(v1_np)
ax[1].set_title("Augmented View 1 (x_i)")
ax[1].axis("off")

v2_np = view2.permute(1, 2, 0).numpy()
v2_np = (v2_np - v2_np.min()) / (v2_np.max() - v2_np.min())
ax[2].imshow(v2_np)
ax[2].set_title("Augmented View 2 (x_j)")
ax[2].axis("off")

plt.suptitle("SimCLR Contrastive Positive Pair Generation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Run Self-Supervised Pre-Training (SimCLR)
!python train_ssl.py --method simclr --backbone resnet18 --epochs 10 --batch_size 16

In [ ]:
# 5. Downstream Defect Classification (Linear Probe vs Supervised Scratch Baseline)
# Evaluating label efficiency at 10% labels
print("--- A. Linear Probe with SSL Pretrained Weights (10% Labels) ---")
!python finetune.py --init simclr --mode linear_probe --split train_10pct --epochs 15 --checkpoint checkpoints/simclr_resnet18_backbone.pth

print("\n--- B. Supervised Baseline from Scratch (10% Labels) ---")
!python finetune.py --init scratch --mode full_finetune --split train_10pct --epochs 15

In [ ]:
# 6. Visualizing Representation Clustering (t-SNE) & Defect Heatmaps (Grad-CAM)
!python evaluate.py --backbone resnet18 --checkpoint checkpoints/simclr_resnet18_backbone.pth

from IPython.display import Image as IPImage, display
if Path("results/tsne_representations.png").exists():
    print("Learned Feature Space (t-SNE Clusters):")
    display(IPImage("results/tsne_representations.png"))

if Path("results/gradcam_sample.png").exists():
    print("Autonomous Structural Defect Localization (Grad-CAM Heatmap):")
    display(IPImage("results/gradcam_sample.png"))